In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/rBMRB')
print("working folder:", os.getcwd())


# Entry filters

Each filter moves the rejected entries out of `bmrb_entries/` into its own folder, so start from a fresh copy of the dump.

## Several protein entities, or no polypeptide

In [ ]:
import sys
MAKESHIFT_ROOT = "../../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
target_dir_1 = os.path.join(base_path, "multi_entities_folders")

os.makedirs(target_dir_1, exist_ok=True)
count = 0

for name in os.listdir(data_path):
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        entities = (entry).sequences()
        if len(entities["ID"].unique()) > 1:
            print(f"Multiple entities in {folder_path}, moving to {target_dir_1}")
            shutil.move(folder_path, os.path.join(target_dir_1, name))
            count += 1
print(f"Total entries with multiple entities: {count}")


In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
target_dir_2 = os.path.join(base_path, "non_polypeptide_folders")

os.makedirs(target_dir_2, exist_ok=True)
count = 0

for name in os.listdir(data_path):
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        entities = (entry).sequences()
        if not entities["Polymer_type"].str.contains("polypeptide").any():
            print(f"No polypeptide in {folder_path}, moving to {target_dir_2}")
            shutil.move(folder_path, os.path.join(target_dir_2, name))
            count += 1
print(f"Total entries without polypeptides: {count}")


## No readable chemical shifts

In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
wrong_cs_read = []

for name in os.listdir(data_path):
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        try:
            cs = get_chem_shifts(entry)
        except:
            print(f"Failed to get chemical shifts for {folder_path}")
            wrong_cs_read.append(name)
            continue
print(f"Total folders with failed chemical shift read: {len(wrong_cs_read)}")

os.makedirs(os.path.join(base_path, "no_cs_read"), exist_ok=True)
no_cs_read_path = os.path.join(base_path, "no_cs_read")
for name in wrong_cs_read:
    folder_path = os.path.join(data_path, name)
    if os.path.isdir(folder_path):
        print(f"Moving {folder_path} to {no_cs_read_path}")
        shutil.move(folder_path, os.path.join(no_cs_read_path, name))

## Modified residues

In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

standard_aa = {
    "ALA", "ARG", "ASN", "ASP", "CYS",
    "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO",
    "SER", "THR", "TRP", "TYR", "VAL"
}

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
invalid_dir = os.path.join(base_path, "modified_residues_folders")
num_invalid = 0
# make sure the target folder exists
os.makedirs(invalid_dir, exist_ok=True)

for name in os.listdir(data_path):
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        cs = get_chem_shifts(entry)
        try:
            unique_residues = set(cs["Comp_ID"].unique())
            invalid_residues = unique_residues - standard_aa

            if invalid_residues:
                raise ValueError(f"Found non-standard residues: {invalid_residues}")
        except Exception as e:
            print(f"Moving {folder_path} to {invalid_dir} due to error: {e}")
            shutil.move(folder_path, os.path.join(invalid_dir, name))
            num_invalid += 1
print(f"Total folders with modified residues: {num_invalid}")

## Non-standard residues in the sequence

In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

standard_aa_1letter = set("ACDEFGHIKLMNPQRSTVWY")

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
invalid_dir = os.path.join(base_path, "nonstand_residues_folders")
num_invalid = 0

for name in os.listdir(data_path):
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        entities = (entry).sequences()
        sequence = entities['Polymer_seq_one_letter_code']
        if any(not set(seq) <= standard_aa_1letter for seq in sequence.dropna()):
            print(f"Found non-standard amino acid in sequence for {folder_path}, moving to {invalid_dir}")
            shutil.move(folder_path, os.path.join(invalid_dir, name))
            num_invalid += 1
print(f"Total folders with non-standard aa in seqs: {num_invalid}")

## Missing backbone shifts

In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
os.makedirs(os.path.join(base_path, "missing_backbone_folders"), exist_ok=True)
missing_backbone_path = os.path.join(base_path, "missing_backbone_folders")
missing_backbone = []

for name in os.listdir(data_path):
    if name.startswith('.'):
        continue
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

        df = get_chem_shifts(ms.NMRStarEntry.from_file(str_files[0]))

        df = df.loc[df.Atom_ID.isin(['H','HA','N','CA','CB','C']) | df.Atom_ID.str.contains('^HA', na=False)]

        if df.empty:
            print(f"No backbone atoms found in {folder_path}, moving to {missing_backbone_path}")
            shutil.move(folder_path, os.path.join(missing_backbone_path, name))
            missing_backbone.append(name)
            continue
print(f"Total folders with missing backbone atoms: {len(missing_backbone)}")


## Solid-state NMR

In [ ]:
import makeshift as ms

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
os.makedirs(os.path.join(base_path, "solidNMR_folders"), exist_ok=True)
solidNMR_path = os.path.join(base_path, "solidNMR_folders")
solidNMR = []

for name in os.listdir(data_path):
    if name.startswith('.'):
        continue
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

        entry = ms.NMRStarEntry.from_file(str_files[0])

        # Primary signal: the structured _Entry_experimental_methods loop
        # (controlled vocabulary, e.g. Method='SOLID-STATE NMR'). Only ~4%
        # of entries have this loop, but when present it's far more
        # reliable than the free-text Experimental_method_subtype tag below
        # -- BMRB deposits that tag inconsistently (we found garbled values
        # like a bare "STATE" that a 'solid' substring check misses).
        methods_df = entry.data_loop("entry_information", "_Entry_experimental_methods")
        is_solid = False
        if not methods_df.empty:
            text = methods_df.reindex(columns=["Method", "Subtype"]).fillna("").astype(str).agg(" ".join, axis=1)
            is_solid = text.str.lower().str.contains("solid").any()
            print(f"Entry {name} has structured experimental methods: {text.tolist()}, is_solid={is_solid}")

        if not is_solid:
            # Fallback: free-text tag, whichever entry_information framecode
            # is actually present (don't guess between 'entry_information'
            # and 'entry_information_1').
            ei_frames = entry.data.get('entry_information', {})
            exp_subtype = next(iter(ei_frames.values()), {}).get('Experimental_method_subtype', '.')
            is_solid = 'solid' in str(exp_subtype).lower()

        if is_solid:
            shutil.move(folder_path, os.path.join(solidNMR_path, name))
            solidNMR.append(name)
            continue

print(f"Total folders with solid state NMR: {len(solidNMR)}")

## Denatured samples

In [ ]:
import makeshift as ms
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
os.makedirs(os.path.join(base_path, "non_native_folders"), exist_ok=True)
non_native_path = os.path.join(base_path, "non_native_folders")
non_native = []

for name in os.listdir(data_path):
    if name.startswith('.'):
        continue
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

        entry = ms.NMRStarEntry.from_file(str_files[0])

        # Check 1: entry.is_denatured() scans each sample's Mol_common_name for
        # denaturant keywords (urea, GdnHCl, guanidine, ...); True means the
        # protein in that sample was not folded/native when the shifts were recorded.
        denatured_table = entry.is_denatured()
        kw_denatured = not denatured_table.empty and denatured_table['denatured'].any()

        # Check 2: the structured _Entity_assembly.Physical_state field (when
        # deposited) states the folding state directly, e.g. 'denatured' vs
        # 'native'. This catches entries Check 1 misses -- e.g. thermally/pH
        # denatured samples where no chemical denaturant appears in the sample
        # component name. Pulled straight from makeshift's parsed output via
        # the generic data_loop() escape hatch (no raw-text regex needed).
        assembly_loop = entry.data_loop("assembly", "_Entity_assembly")
        phys_denatured = (
            "Physical_state" in assembly_loop.columns
            and assembly_loop["Physical_state"].astype(str).str.lower().str.contains("denatur", na=False).any()
        )

        if kw_denatured or phys_denatured:
            reasons = []
            if kw_denatured:
                reasons.append("Mol_common_name keyword")
            if phys_denatured:
                reasons.append("Entity_assembly.Physical_state")
            print(f"Non-native ({', '.join(reasons)}) sample found in {folder_path}, moving to {non_native_path}")
            shutil.move(folder_path, os.path.join(non_native_path, name))
            non_native.append(name)
            continue
print(f"Total folders with non-native (denatured) protein: {len(non_native)}")


## Longer than 500 residues

In [ ]:
import makeshift as ms
import os
import glob
import shutil

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
invalid_dir = os.path.join(base_path, "long_seq_folders")
num_invalid = 0
os.makedirs(invalid_dir, exist_ok=True)

for name in os.listdir(data_path):
    if name.startswith('.'):
        continue
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        entry = ms.NMRStarEntry.from_file(str_files[0])
        entities = entry.sequences()

        for sequence in entities['Polymer_seq_one_letter_code'].dropna():
            if len(sequence) > 500:
                print(f"Entry {name} has sequence length {len(sequence)}, moving to {invalid_dir}")
                shutil.move(folder_path, os.path.join(invalid_dir, name))
                num_invalid += 1
                break

print(f"Total folders removed due to excessive sequence length: {num_invalid}")